# 3.3 Human-in-the-Loop (HITL) — Pausing Agents for Approval

## What you will learn in this notebook

- What **Human-in-the-Loop** means and why it's essential for high-stakes tools
- How `HumanInTheLoopMiddleware` pauses the agent before executing specific tools
- How to inspect the **interrupt payload** to see what the agent wants to do
- How to **approve**, **reject**, or **edit** a pending tool call and resume execution

---

### The problem: agents acting autonomously on high-stakes tools

Some tool calls are safe to run automatically — reading data, looking things up. Others are **irreversible or high-stakes** — sending an email, charging a payment, deleting a record. Letting the agent execute these without oversight is dangerous.

**Human-in-the-Loop (HITL)** adds a checkpoint:

```
WITHOUT HITL:                          WITH HITL:

User: "Send a reply to John"           User: "Send a reply to John"
Agent: [reads email]                   Agent: [reads email]
Agent: [sends email immediately] ←DANGEROUS  Agent: PAUSES → shows draft to user
                                       User: reviews draft
                                       User: approves / rejects / edits
                                       Agent: [sends email with approved content]
```

### How it works technically

LangGraph uses **interrupt** — a mechanism that suspends the agent graph at a specific point and saves state. The graph stays paused until you resume it with a `Command`. This is different from the agent simply returning — the graph is truly suspended mid-execution.

```
agent.invoke() → graph runs → tool call decision → INTERRUPT (pause here)
                                                         ↓
                                              response has '__interrupt__' key
                                                         ↓
                                              user inspects the draft
                                                         ↓
agent.invoke(Command(resume={...})) → graph resumes → tool executes → final answer
```

In [4]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [5]:
# ============================================================
# CELL 2: Define Read and Send Email Tools
# ============================================================
# Two tools with very different risk profiles:
#
# read_email (LOW risk):
#   → Reads from state — no side effects, completely safe
#   → We will configure HITL to NOT interrupt on this tool
#
# send_email (HIGH risk):
#   → Sends an actual email — irreversible side effect
#   → We will configure HITL to INTERRUPT on this tool
#   → The agent proposes the email body; the human approves/rejects/edits
#
# read_email uses runtime.state["email"] — the email text comes from
# the custom EmailState field, not from the user's message.
# This is the state-reading pattern from notebook 2.2.
#
# send_email is intentionally fake (returns a string, doesn't actually
# send) — in production this would call an email API (Gmail, SendGrid, etc.)

from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    return runtime.state["email"]   # Read from custom state field

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    return f"Email sent"  # Fake — replace with real email API in production

In [6]:
# ============================================================
# CELL 3: Create the Agent With HITL Middleware
# ============================================================
# EmailState extends AgentState with one custom field:
#   email: str  → the email content the agent will read via read_email
#   This is injected at invoke() time in the input dict.
#
# HumanInTheLoopMiddleware parameters:
#
#   interrupt_on  → a dict mapping tool names to booleans
#     True  = PAUSE before executing this tool (require human approval)
#     False = execute this tool automatically (no pause)
#
#     "read_email": False  → safe to run automatically
#     "send_email": True   → pause and wait for human approval
#
#   description_prefix → a string prepended to the interrupt message
#     This appears in the '__interrupt__' payload and is shown to the
#     human reviewer so they understand WHY execution is paused.
#
# checkpointer=InMemorySaver() is REQUIRED for HITL.
# The agent graph must persist its state while paused so it can be
# resumed in a later .invoke() call. Without a checkpointer, the
# paused state would be lost and resumption would fail.

from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str   # The email content to read

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),   # REQUIRED for HITL — persists paused state
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,   # Auto-execute — safe read operation
                "send_email": True,    # PAUSE — requires human approval
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [10]:
# ============================================================
# CELL 4: Invoke — Agent Will Pause Before Sending
# ============================================================
# The input dict has two keys:
#   "messages"  → the user's instruction
#   "email"     → the email content injected into EmailState
#
# The agent will:
#   1. Read the instruction: "read email and send a response"
#   2. Call read_email() — auto-executed (interrupt_on=False)
#   3. Compose a reply based on the email content
#   4. Attempt to call send_email() — INTERRUPTED (interrupt_on=True)
#   5. Return with a '__interrupt__' key in the response dict
#
# The response will NOT contain a final AIMessage yet — the graph
# is suspended mid-execution waiting for your decision.

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "5"}}  # Thread ID for resuming later

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."  # Injected into EmailState
    },
    config=config
)

In [8]:
from pprint import pprint
for message in response['messages']:
    print("===============")
    pprint(message)

HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='316603c9-f774-4a8a-8b10-f3f0c080d806')
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 167, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E10pmxVpfmysfpomMK5ZMRbR8OOJG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f5956-12d0-7890-be78-759d09ee6cef-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_dHgDYaVKva55xpbqZjhZk0Fg', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

In [31]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John,\n\nNo problem—thanks for the heads up. I'm happy to reschedule. Are you available tomorrow at 11:00 AM or 2:00 PM? If those times don’t work, please share a couple of alternatives and I’ll confirm the best option.\n\nBest regards,\nSeán"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nNo problem—thanks for the heads up. I\'m happy to reschedule. Are you available tomorrow at 11:00 AM or 2:00 PM? If those times don’t work, please share a couple of alternatives and I’ll confirm the best option.\\n\\nBest regards,\\nSeán"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='311a4d191d98e0068f41148e409baf19')]


In [32]:
# ============================================================
# CELL 7: Extract the Proposed Email Body
# ============================================================
# Drill into the interrupt structure to get exactly what the agent
# wants to send. This is what you'd show a human reviewer:
#
# response['__interrupt__']     → list of Interrupt objects
#   [0]                         → first interrupt (only one here)
#   .value                      → the interrupt data dict
#   ['action_requests']         → list of pending tool calls
#   [0]                         → first pending call (send_email)
#   ['args']                    → the arguments dict
#   ['body']                    → the 'body' parameter of send_email
#
# In a web app, you'd display this body in a UI for the human to review.

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads up. I'm happy to reschedule. Are you available tomorrow at 11:00 AM or 2:00 PM? If those times don’t work, please share a couple of alternatives and I’ll confirm the best option.

Best regards,
Seán


---
## Section 2 — Three Ways to Respond to an Interrupt

Once the agent is paused, you have three choices:

| Decision | What happens |
|---|---|
| **`approve`** | Execute the tool exactly as the agent planned |
| **`reject`** | Don't execute the tool; tell the agent why so it can try again |
| **`edit`** | Execute the tool but with different arguments that YOU specify |

All three use `Command(resume={"decisions": [...]})` — the same config (thread_id) resumes the paused graph.

In [9]:
# ============================================================
# CELL 8: Option A — Approve (Execute As-Is)
# ============================================================
# Command(resume=...) tells LangGraph to resume the paused graph.
#
# decisions: [{"type": "approve"}]
#   → Execute the tool with exactly the arguments the agent chose.
#   → The agent drafted the email body; we approve it as-is.
#
# config=config (same thread_id="1")
#   → CRITICAL: must use the same thread_id as the original invoke().
#   → LangGraph loads the paused graph state from InMemorySaver
#     using this thread_id and resumes from the interrupt point.

from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}   # Run send_email as-is
    ), 
    config=config   # Same thread_id — resumes the paused graph
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='316603c9-f774-4a8a-8b10-f3f0c080d806'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 167, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E10pmxVpfmysfpomMK5ZMRbR8OOJG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f5956-12d0-7890-be78-759d09ee6cef-0', tool_calls

---
### Option B — Reject

In [11]:
# ============================================================
# CELL 9: Option B — Reject (Don't Execute; Give Feedback)
# ============================================================
# decisions: [{"type": "reject", "message": "..."}]
#   → The tool is NOT executed.
#   → The rejection message is fed back to the agent as context.
#   → The agent reads the message and attempts to draft a new version.
#   → This typically triggers ANOTHER interrupt (the agent writes
#     a new email draft that must be approved again).
#
# The message field is critical — without it, the agent doesn't
# know WHY it was rejected and may re-draft the same email.
# Be specific: "Too informal", "Missing subject line", "Wrong tone".

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "No please sign off - Your merciful leader, Seán."  # Explanation for the agent
                }
            ]
        }
    ), 
    config=config   # Same thread_id — resumes the paused graph
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads '
                                                                          'up. '
                                                                          'I '
                                                                          'can '
                                                                          'reschedule. '
          

In [34]:
# ============================================================
# CELL 10: After Reject — Agent Re-Drafts, Interrupts Again
# ============================================================
# The agent read the rejection reason and wrote a new email draft.
# The middleware detected another send_email call and paused again.
# We can see the new (revised) draft here.

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads up. I'm happy to reschedule. Are you available tomorrow at 11:00 AM or 2:00 PM? If those times don’t work, please share a couple of alternatives and I’ll confirm the best option.

Your merciful leader,
Seán


---
### Option C — Edit

In [12]:
# ============================================================
# CELL 11: Option C — Edit (Override the Tool Arguments)
# ============================================================
# decisions: [{"type": "edit", "edited_action": {...}}]
#   → The tool IS executed, but with YOUR arguments, not the agent's.
#   → You completely override what the agent planned to send.
#
# edited_action fields:
#   "name" → the tool to call (usually the same tool that was interrupted)
#   "args" → the arguments to pass (you write these yourself)
#
# In this example, the agent drafted a polite response but we
# override it with a very different message — same tool, different args.
#
# Use cases for edit:
#   - The agent's draft is 90% right but needs small corrections
#   - Compliance requires specific legal language the agent doesn't know
#   - The human wants to personalise a templated reply
from langgraph.types import Command
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",                    # Same tool
                        "args": {"body": "This is the last straw, you're fired!"},  # OUR content
                    }
                }
            ]
        }
    ), 
    config=config
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads '
                                                                          'up. '
                                                                          'I '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'What '
                

In [35]:
from pprint import pprint

for message in response['messages']:
    print("===============")
    pprint(message)

HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='eacdb0fe-a653-4f6d-b0d4-71e178afd333')
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 467, 'prompt_tokens': 167, 'total_tokens': 634, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DPJGIS0f0HuVUTwR7Af1TQ9AZVGKA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d41c0-464f-7f32-94a7-8fdab4cd02da-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_6ny3vFIBnxWtkAjSobDIW4D2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

In [27]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'Yes, '
                                                                          'we '
                                                                          'can '
                                                                          'reschedule. '
            

---
## Summary

| Concept | Key takeaway |
|---|---|
| **HITL** | Pause the agent before high-stakes tool calls — human reviews and decides |
| **`HumanInTheLoopMiddleware`** | Configure per-tool: `True` = pause, `False` = auto-execute |
| **`checkpointer` required** | Must persist state so the graph can be resumed after pausing |
| **`response['__interrupt__']`** | Signals the graph is paused; contains the pending tool call details |
| **`approve`** | Execute the tool exactly as the agent planned |
| **`reject` + `message`** | Don't execute; agent reads the message and re-drafts |
| **`edit` + `edited_action`** | Execute the tool with your arguments instead of the agent's |
| **Same `config` to resume** | The thread_id is how LangGraph finds and resumes the paused graph |

### HITL decision flowchart

```
Agent pauses at send_email
        │
  Review the draft
        │
  ┌─────┼────────────┐
  ▼     ▼            ▼
approve reject      edit
  │     │ (agent    │ (your
  │     │  re-drafts)│  args)
  ▼     ▼            ▼
send  [new pause]   send with
                    your content
```